# **Dependencies**

In [ ]:
!pip install ultralytics
!pip install roboflow

# **Imports**

In [ ]:
from ultralytics import YOLO
from roboflow import Roboflow
import numpy as np
import cv2
import os

# **Data Download**

## Train Data

In [ ]:
# Normal train-set
# rf = Roboflow(api_key="cIEHAX4VyYolY0CtAbnT")
# project = rf.workspace("sam-on-building-detection").project("mt-seg-ann-1024_compressed_cv2")
# version = project.version(19)
# dataset = version.download("yolov9") # Downloaded dataset will be in yolov9 suitable format

rf = Roboflow(api_key="cIEHAX4VyYolY0CtAbnT")
project = rf.workspace("building-segmentation-traintest-bcha5").project("mt-seg-ann-1024_compressed_cv2")
version = project.version(22)
dataset = version.download("yolov8") # Downloaded dataset will be in yolov8 suitable format

In [ ]:
# Augmented for OBB training (v8 obb format export)
rf = Roboflow(api_key="cIEHAX4VyYolY0CtAbnT")
project = rf.workspace("building-segmentation-traintest-bcha5").project("mt-seg-ann-1024_compressed_cv2")
version = project.version(21)
dataset = version.download("yolov8-obb")


## Test Data

In [ ]:
!wget https://huggingface.co/datasets/abturjo/dengue_test_data/resolve/main/tiles_JPG_25p.zip # Tiles generated with 25% overlap
!wget https://huggingface.co/datasets/abturjo/dengue_test_data/resolve/main/tiles_JPG_50p.zip # Tiles generated with 50% overlap
!wget https://huggingface.co/datasets/abturjo/dengue_test_data/resolve/main/tiles_JPG_75p.zip # Tiles generated with 75% overlap

# **Weights download**

## Pretrained Model Weights
- From YOLO \
- Some weights require explicit download (f.e. yolov11m weights which are yet not added in ultralytics repository during this project)

In [ ]:
!wget 'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11m-seg.pt'
!wget 'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11s-seg.pt'

## Our Trained Model Weights
- Inference on our dataset

In [ ]:
!wget 'https://huggingface.co/abturjo/dengue_trained_weights/resolve/main/best_v8m.pt'
!wget 'https://huggingface.co/abturjo/dengue_trained_weights/resolve/main/best_v11m.pt'

# **Model : YOLO series**

## Train

In [ ]:
# 'yolov11m-seg.pt' is required to be separately downloaded, but 'yolov8m-seg.pt' would work with only the constructor command
model = YOLO('yolo11m-seg.pt')  # You can choose 'yolov8m-seg.pt', etc., based on your preference

# model.train(data=f'{dataset.location}/data.yaml', epochs=100, batch=8, imgsz=1024)
model.train(data='/content/onestepDataset/data.yaml', epochs=100, batch=8, imgsz=1024)


## Validation

In [ ]:
results = model.val()

## Test

In [ ]:
# Predict and visualize results on test images
test_images_path = '/content/tiles_JPG_25p'

model = YOLO('/content/best_v11m.pt')
results = model.predict(source=test_images_path, imgsz=1024, conf=0.8, save=True, save_txt=True, save_conf=True) # Use save=True to save the results

# **Single Mask Save**

In [ ]:
# Single Mask Saves
image_height = 2048
image_width = 2048
save_path = '/content/yolov8m_inference'

os.makedirs(save_path, exist_ok=True)

for index, result in enumerate(results):
    folder_name = os.path.splitext(os.path.basename(result.path))[0]
    folder_path = os.path.join(save_path, folder_name)
    os.makedirs(folder_path, exist_ok=True)

    background = np.zeros((image_height, image_width), dtype=np.uint8)

    if result.masks is None:
        print(os.path.basename(result.path), ' file has no masks')
        continue

    masks = result.masks.xy

    for i, mask in enumerate(masks):
        points = np.array(mask, dtype=np.int32).reshape((-1, 1, 2))
        cv2.fillPoly(background, [points], 255)
        individual_mask_save_path = os.path.join(folder_path, f'mask_{i+1}.png')
        cv2.imwrite(individual_mask_save_path, background)
        background = np.zeros((image_height, image_width), dtype=np.uint8)

    print(f'Processed file: {os.path.basename(result.path)}')


# **Masks per image save**

In [ ]:
image_height = 2048
image_width = 2048
save_path = '/content/yolov11m_inference'

os.makedirs(save_path, exist_ok=True)

for index, result in enumerate(results):
    folder_name = os.path.splitext(os.path.basename(result.path))[0]
    folder_path = os.path.join(save_path, folder_name)

    # Combined mask image save path
    combined_mask_save_path = folder_path + '.png'

    # Create a single background to hold all masks for the image
    background = np.zeros((image_height, image_width), dtype=np.uint8)

    if result.masks is None:
        print(os.path.basename(result.path), ' file has no masks')
        cv2.imwrite(combined_mask_save_path, background)
        continue

    masks = result.masks.xy

    # Loop through all masks and draw them on the same background
    for mask in masks:
        points = np.array(mask, dtype=np.int32).reshape((-1, 1, 2))
        cv2.fillPoly(background, [points], 255)

    # Save the combined mask image
    cv2.imwrite(combined_mask_save_path, background)

    print(f'Processed file: {os.path.basename(result.path)}')
    print("Masks found -->", len(masks))


# **Saving all masks with confidence score**

In [ ]:
import numpy as np

masks_save_dir = '/content/labels_8m_75'
os.makedirs(masks_save_dir, exist_ok=True)

for index, result in enumerate(results):
    classes = result.boxes.cls
    masks = result.masks
    confs = result.boxes.conf
    filename = result.path.split('/')[-1]
    # pos = filename.strip().split('_')[2].split('-')[1:3]
    pos = filename.split('_')[3:5]

    if masks is not None:
        for i, mask in enumerate(masks):
            print(mask)
            mask_filename = f"{index+1}_{pos[0]}-{pos[1]}.txt"
            mask_filepath = os.path.join(masks_save_dir, mask_filename)

            # Flatten mask coordinates and convert to list
            mask = mask.xy[0].flatten().tolist()

            # Append confidence score to the end of the mask list
            mask.append(confs[i].item())
            # Prepend the class_id
            mask.insert(0, int(classes[i].item()))

            # Convert to a space-separated string
            mask_string = ' '.join(map(str, mask))

            # Write to file
            with open(mask_filepath, 'a') as f:
                f.write(f"{mask_string}\n")



# **File Download Section**

In [ ]:
!zip -r "test_result_yolo" "/content/runs/segment/predict2" "/content/runs/segment/predict3"